# 05 — Comparaison finale sur le jeu test

Les deux systèmes sont évalués sur les 20 questions du jeu **test**, qui n'ont servi à aucun réglage : 12 questions factuelles, 4 multi-articles et 4 hors corpus. Le code est figé (même commit pour les deux évaluations), avec le même modèle, le même index, au plus 5 passages par réponse et le même LLM-juge. Les 32 réponses notées ont été relues une à une (`revue_humaine.csv`).

In [1]:
import json

import numpy as np
import pandas as pd

from assistant_rag.config import RESULTS_DIR
from assistant_rag.evaluation.judge import VERDICT_SCORES
from assistant_rag.evaluation.metrics import score_records, summarize
from assistant_rag.evaluation.runner import load_records

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.precision", 2)

SYSTEMS = {"RAG simple": "simple", "Agents": "agents"}


def load_run(run):
    records = load_records(RESULTS_DIR / run / "answers.jsonl")
    scores = score_records(records).set_index("id")
    review = pd.read_csv(RESULTS_DIR / run / "revue_humaine.csv").set_index("id")
    human = scores["correctness"].copy()
    human.loc[review.index] = review["verdict_humain"].map(VERDICT_SCORES)
    scores["human"] = human
    return records, scores, review


test = {label: load_run(f"{prefix}-test") for label, prefix in SYSTEMS.items()}
dev = {label: load_run(f"{prefix}-v2-dev") for label, prefix in SYSTEMS.items()}

commits = {
    label: json.loads((RESULTS_DIR / f"{prefix}-test" / "config.json").read_text())["git_commit"]
    for label, prefix in SYSTEMS.items()
}
commits

{'RAG simple': 'd10f54c', 'Agents': 'd10f54c'}

## 1. Métriques automatiques

In [2]:
key_metrics = [
    "Rappel@k",
    "Précision des citations",
    "Rappel des citations",
    "Exactitude",
    "Refus à tort",
    "Refus correct",
    "Latence moyenne (s)",
    "Appels au LLM",
]
pd.concat({label: summarize(records) for label, (records, _, _) in test.items()}, axis=1).swaplevel(axis=1)[key_metrics]

Rappel@k        Précision des citations         \
               RAG simple Agents              RAG simple Agents   
Factuelles            1.0    1.0                    0.93   0.79   
Multi-articles        1.0    1.0                    0.90   1.00   
Hors corpus           NaN    NaN                     NaN    NaN   
Ensemble              1.0    1.0                    0.92   0.84   

               Rappel des citations        Exactitude        Refus à tort  \
                         RAG simple Agents RAG simple Agents   RAG simple   
Factuelles                      1.0   0.92       0.88   0.83          0.0   
Multi-articles                  1.0   1.00       0.75   0.88          0.0   
Hors corpus                     NaN    NaN        NaN    NaN          NaN   
Ensemble                        1.0   0.94       0.84   0.84          0.0   

                      Refus correct        Latence moyenne (s)         \
               Agents    RAG simple Agents          RAG simple Agents   
Factuelles        0.0           NaN    NaN                9.54  11.59   
Multi-articles    0.0           NaN    NaN                6.41  30.21   
Hors corpus       NaN           1.0    1.0                1.94   2.66   
Ensemble          0.0           1.0    1.0                7.39  13.53   

               Appels au LLM         
                  RAG simple Agents  
Factuelles               1.0    4.0  
Multi-articles           1.0    5.0  
Hors corpus              1.0    2.0  
Ensemble                 1.0    3.8

La recherche est parfaite pour les deux systèmes : tous les articles attendus sont retrouvés, y compris pour les questions multi-articles. L'avantage du chercheur observé sur le jeu dev ne se manifeste donc pas ici. Les 4 questions hors corpus sont refusées par les deux systèmes.

## 2. Exactitude après relecture

In [3]:
def human_table(runs):
    table = {}
    for label, (_, scores, review) in runs.items():
        answerable = scores[scores["type"] != "hors_corpus"]
        column = answerable["human"].groupby(answerable["type"]).mean()
        column["ensemble"] = answerable["human"].mean()
        column["accord juge / relecture"] = (review["verdict_juge"] == review["verdict_humain"]).mean()
        table[label] = column
    return pd.DataFrame(table)


human_table(test)

,RAG simple,Agents
type,,
factuelle,0.96,0.88
multi_articles,0.88,0.75
ensemble,0.94,0.84
accord juge / relecture,0.81,0.75


In [4]:
verdicts = pd.DataFrame({label: review["verdict_humain"] for label, (_, _, review) in test.items()})
differences = verdicts[verdicts["RAG simple"] != verdicts["Agents"]]
differences.join(test["Agents"][2]["commentaire"].rename("commentaire (agents)"))

,RAG simple,Agents,commentaire (agents)
id,,,
aiact-03,correct,partiel,"Affirme à tort que l'obligation ne vise pas explicitement les déployeurs, alors que l'article 4 les nomme."
aiact-13,correct,partiel,"Délais inversés : « deux jours ou dix jours respectivement » pour le décès et l'infraction de grande ampleur, au lieu de dix et deux jours."
multi-09,partiel,correct,NaN
multi-12,correct,incorrect,Montant maximal de l'AI Act faux : 15 000 000 EUR ou 3 % au lieu de 35 000 000 EUR ou 7 %.


Sur le jeu test, les agents font **moins bien** que le RAG simple (0,84 contre 0,94). Les deux systèmes retrouvent les mêmes passages : les écarts viennent tous de la rédaction.

- **aiact-03** : le chercheur sépare fournisseurs et déployeurs ; le rédacteur affirme ensuite que l'obligation ne vise pas les déployeurs, alors que l'article 4 les nomme.
- **aiact-13** : en résumant les délais, le rédacteur inverse ceux applicables au décès et à l'infraction de grande ampleur.
- **multi-12** : le rédacteur retient l'amende de 15 000 000 EUR au lieu du maximum de 35 000 000 EUR.
- **multi-09** : à l'inverse, les agents décrivent correctement les obligations des déployeurs, que le RAG simple traite de façon incomplète.

## 3. Incertitude

Avec 16 questions notées par système sur le jeu test, une question correspond à 6 points d'exactitude. Deux mesures permettent de situer l'écart observé.

**Variabilité d'un même système.** Sur GPU, la génération n'est pas parfaitement déterministe. Le RAG simple a été évalué deux fois sur le jeu dev avec le même code :

In [5]:
first = {r["id"]: r["answer"].strip() for r in load_records(RESULTS_DIR / "simple-dev" / "answers.jsonl")}
second = {r["id"]: r["answer"].strip() for r in load_records(RESULTS_DIR / "simple-v2-dev" / "answers.jsonl")}
changed = sum(first[qid] != second[qid] for qid in first)
print(f"Réponses différentes d'une exécution à l'autre : {changed} / {len(first)}")

Réponses différentes d'une exécution à l'autre : 19 / 40


**Intervalle de confiance.** L'écart d'exactitude relue entre les deux systèmes est estimé sur les 48 questions avec réponse des jeux dev et test réunis, par un bootstrap apparié : les mêmes questions sont tirées pour les deux systèmes.

In [6]:
def paired_scores(runs):
    simple = runs["RAG simple"][1]
    agents = runs["Agents"][1]
    ids = simple.index[simple["type"] != "hors_corpus"]
    return simple.loc[ids, "human"].to_numpy(), agents.loc[ids, "human"].to_numpy()


simple_scores, agents_scores = (
    np.concatenate(parts) for parts in zip(paired_scores(dev), paired_scores(test), strict=True)
)
differences_per_question = agents_scores - simple_scores

rng = np.random.default_rng(0)
samples = rng.integers(0, len(differences_per_question), size=(10_000, len(differences_per_question)))
bootstrap = differences_per_question[samples].mean(axis=1)
low, high = np.percentile(bootstrap, [2.5, 97.5])

pd.Series(
    {
        "questions": len(differences_per_question),
        "exactitude RAG simple": simple_scores.mean(),
        "exactitude agents": agents_scores.mean(),
        "écart (agents − simple)": differences_per_question.mean(),
        "IC 95 % bas": low,
        "IC 95 % haut": high,
    }
).round(3).to_frame("valeur")

,valeur
questions,48.00
exactitude RAG simple,0.91
exactitude agents,0.89
écart (agents − simple),-0.02
IC 95 % bas,-0.10
IC 95 % haut,0.06


L'intervalle de confiance contient zéro : sur l'ensemble des questions disponibles, **aucun des deux systèmes n'est significativement plus exact que l'autre**.

## 4. Signal du vérificateur

In [7]:
_, agents_scores_test, agents_review = test["Agents"]
records_test = {r["id"]: r for r in test["Agents"][0]}
flags = (
    pd.Series({qid: records_test[qid]["verified"] for qid in agents_review.index})
    .map({True: "validée", False: "non validée"})
    .rename("vérificateur")
)
pd.crosstab(flags, agents_review["verdict_humain"], margins=True, margins_name="total")

verdict_humain,correct,incorrect,partiel,total
vérificateur,,,,
non validée,1,1,1,3
validée,11,0,2,13
total,12,1,3,16


Comme sur le jeu dev, les réponses non validées par le vérificateur sont plus souvent fausses ou incomplètes (2 sur 3, contre 2 sur 13 parmi les réponses validées).

## 5. Coût

In [8]:
pd.DataFrame(
    {
        label: {
            "latence médiane (s)": scores["latency"].median(),
            "latence moyenne (s)": scores["latency"].mean(),
            "appels au LLM par question": scores["llm_calls"].mean(),
        }
        for label, (_, scores, _) in test.items()
    }
)

,RAG simple,Agents
latence médiane (s),3.37,6.87
latence moyenne (s),7.39,13.53
appels au LLM par question,1.00,3.80


## 6. Conclusions

**Résultat.** Sur le jeu test, le RAG simple obtient une exactitude relue de 0,94 et le système multi-agents de 0,84. Sur les 48 questions des jeux dev et test réunis, l'écart n'est pas significatif (l'intervalle de confiance à 95 % contient zéro), et il est du même ordre que la variabilité d'un même système d'une exécution à l'autre. Le système multi-agents coûte environ 4 fois plus d'appels au LLM et environ deux fois plus de temps de réponse.

**Ce que les agents apportent réellement.**

- Le **chercheur** améliore la couverture des questions qui combinent plusieurs articles lorsque la recherche simple en manque un (jeu dev : rappel de 81 % à 94 %). Sur le jeu test, la recherche simple retrouvait déjà tout.
- Le **vérificateur** ne corrige pas les réponses de façon fiable avec un modèle de 4 milliards de paramètres, mais il signale les réponses douteuses : c'est un indicateur utile à afficher à l'utilisateur.
- La **décomposition** a un coût : en découpant la question, le rédacteur traite chaque aspect séparément et commet des erreurs de synthèse (aiact-03, aiact-13).

**Choix retenu pour l'application.** Le RAG simple est le mode par défaut : il est aussi exact, plus rapide et moins coûteux. Le mode multi-agents reste disponible pour les questions qui combinent plusieurs règles, avec l'indication « réponse non vérifiée » lorsque le vérificateur ne l'a pas validée.

**Limites.**

- Jeu d'évaluation réduit (48 questions avec réponse) : seuls des écarts importants seraient détectables.
- Un seul modèle de génération, de petite taille, imposé par l'exécution locale sur processeur.
- Relecture humaine faite par une seule personne ; le LLM-juge n'est d'accord avec elle que dans 74 à 84 % des cas selon les évaluations.
- Génération non déterministe sur GPU : chaque système n'a été évalué qu'une fois sur le jeu test.

**Pistes.** Utiliser un modèle plus grand pour le rédacteur et le vérificateur, mesurer une version des agents sans vérificateur, et agrandir le jeu de questions. Ces expériences sont suivies avec MLflow à l'étape suivante.